# 03. Training — Supervised Learning (CE Loss, 10 Epochs)
# 03. 학습 — 지도 학습 (CE Loss, 10 Epochs)

**Goal / 목표**
- Train a Grayspot Level classifier using CrossEntropyLoss for 10 epochs
- CrossEntropyLoss를 사용하여 10 Epochs 동안 Grayspot Level 분류기를 학습한다
- This is a Naive Baseline — no Contrastive Learning, no Swing Architecture
- 이 노트북은 Naive Baseline 용으로 Contrastive Learning 및 Swing Architecture 없이 실행한다

**Folder Structure / 폴더 구조**
```
data/labeled/{channel}/{level}/*.png
예시 / Example: data/labeled/C/2/scan_001_C_0007.png
```

**Execution Order / 실행 순서**
1. Import libraries / 라이브러리 임포트
2. Path & settings / 경로 및 설정
3. Build dataset and dataloader / 데이터셋 및 DataLoader 구성
4. Build model / 모델 구성
5. Training loop (10 epochs) / 학습 루프 (10 에폭)
6. Verify results / 결과 확인

---
## 1. Import Libraries / 라이브러리 임포트

In [1]:
import csv
import time
import random
import numpy as np
import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision import models
from pathlib import Path

# 재현성을 위한 시드 고정 / Fix seed for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# 디바이스 설정 / Device setup
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)

print("Libraries loaded / 라이브러리 로드 완료")
print(f"PyTorch version / 버전: {torch.__version__}")
print(f"Device / 디바이스: {device}")

Libraries loaded / 라이브러리 로드 완료
PyTorch version / 버전: 2.11.0
Device / 디바이스: mps


---
## 2. Path & Settings / 경로 및 설정

> config.yaml 없이 직접 경로와 값을 지정한다.
> Paths and values are set directly without config.yaml.

In [2]:
ROOT_DIR = Path("../..").resolve()  # CMYK_MAIN/
LABELED_DIR = ROOT_DIR / "data_set" / "labeled"  # 라벨링 폴더 / Labeled folder
MODEL_DIR = ROOT_DIR / "data_set" / "models"  # 모델 저장 폴더 / Model save folder
REPORT_DIR = ROOT_DIR / "data_set" / "reports"  # 리포트 폴더 / Report folder

CHANNELS = ["Y", "M", "C", "K"]
NUM_LEVELS = 6  # Level 0 ~ 5
IMAGE_SIZE = 128  #  기준 크기 /  standard size
BACKBONE_NAME = "efficientnet_b0"  # 'efficientnet_b0' | 'resnet50'

# 학습 하이퍼파라미터 / Training hyperparameters
EPOCHS = 30  # 학습 에폭 수 / Number of epochs
BATCH_SIZE = 16  # 배치 크기 / Batch size
LEARNING_RATE = 1e-4  # 초기 학습률 / Initial learning rate
WEIGHT_DECAY = 1e-4  # L2 정규화 / L2 regularization
TARGET_CHANNEL = "C"  # 테스트할 채널 / Channel to test — Y / M / C / K
PHASE0_BACKBONE = (
    ROOT_DIR / "data_set" / "models" / f"phase0_backbone_{TARGET_CHANNEL}.pt"
)  # Phase0_Backbone

MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Backbone / 백본: {BACKBONE_NAME}")
print(f"Target channel / 대상 채널: {TARGET_CHANNEL}")
print(f"Epochs: {EPOCHS} | Batch size: {BATCH_SIZE} | LR: {LEARNING_RATE}")
print(f"Labeled dir / 라벨 경로: {LABELED_DIR}")
print(f"Phase 0 backbone exists: {PHASE0_BACKBONE.exists()}")
print(f"Path: {PHASE0_BACKBONE}")

Backbone / 백본: efficientnet_b0
Target channel / 대상 채널: C
Epochs: 30 | Batch size: 16 | LR: 0.0001
Labeled dir / 라벨 경로: /Users/yangjin-hyeong/Desktop/CMYK_MAIN/data_set/labeled
Phase 0 backbone exists: True
Path: /Users/yangjin-hyeong/Desktop/CMYK_MAIN/data_set/models/phase0_backbone_C.pt


---
## 3. Build Dataset and DataLoader / 데이터셋 및 DataLoader 구성

- Load labeled patches from `data/labeled/{channel}/{level}/`
- `data/labeled/{channel}/{level}/` 에서 라벨링된 패치를 로드한다
- Split 70 / 15 / 15 (Train / Val / Test)
- 70 / 15 / 15 비율로 분할한다

In [3]:
import random
from collections import defaultdict, Counter


class CMYKDataset(Dataset):
    """
    기준 Dataset — data_set/labeled/{channel}/{level}/ 폴더 구조 기반.
    standard Dataset — based on data_set/labeled/{channel}/{level}/ folder structure.
    """

    def __init__(
        self,
        channel: str,
        split: str = "train",
        augment: bool = True,
        oversample: bool = True,
    ):
        """
        Args:
            channel:   \"Y\" | \"M\" | \"C\" | \"K\"
            split:     \"train\" | \"val\" | \"test\"
            augment:   학습 시 증강 적용 여부 / Apply augmentation during training
            oversample: 학습 시 오버샘플링 적용 여부 / Apply oversampling during training
        """
        self.augment = augment and (split == "train")
        self.exts = {".png", ".jpg", ".jpeg", ".tiff", ".tif"}
        all_samples = []  # [(image_path, level), ...]

        # data_set/labeled/{channel}/{level}/ 에서 샘플 수집
        # Collect samples from data_set/labeled/{channel}/{level}/
        channel_dir = LABELED_DIR / channel
        for level in range(NUM_LEVELS):
            level_dir = channel_dir / str(level)  # 0, 1, 2, 3, 4, 5
            if not level_dir.exists():
                continue
            for img_path in sorted(level_dir.glob("*")):
                if img_path.suffix.lower() in self.exts:
                    all_samples.append((img_path, level))

        # Stratified Split — 레벨별 비율 유지 / Maintain level distribution per split
        level_groups = defaultdict(list)
        for sample in all_samples:
            level_groups[sample[1]].append(sample)

        train_samples, val_samples, test_samples = [], [], []
        for lv, items in level_groups.items():
            random.shuffle(items)
            n = len(items)
            n_train = max(1, int(n * 0.70))
            n_val = max(1, int(n * 0.15))
            train_samples.extend(items[:n_train])
            val_samples.extend(items[n_train : n_train + n_val])
            test_samples.extend(items[n_train + n_val :])

        if split == "train":
            self.samples = train_samples

            # Oversampling — 가장 많은 클래스 수 기준으로 맞춤
            # Oversample to match the largest class count
            if oversample and len(self.samples) > 0:
                level_counts = Counter([lv for _, lv in self.samples])
                max_count = max(level_counts.values())

                oversampled = []
                for level in range(NUM_LEVELS):
                    level_samples = [(p, lv) for p, lv in self.samples if lv == level]
                    if len(level_samples) == 0:
                        continue
                    # 부족한 만큼 랜덤 복제 / Replicate randomly to fill up
                    while len(level_samples) < max_count:
                        level_samples.append(random.choice(level_samples))
                    oversampled.extend(level_samples)

                self.samples = oversampled
                print(
                    f"[{channel}] Oversampling 적용 / Applied: "
                    f"{len(train_samples)}개 → {len(self.samples)}개"
                )

        elif split == "val":
            self.samples = val_samples
        else:  # test
            self.samples = test_samples

        # 분할 결과 확인 / Check split results
        split_counts = Counter([lv for _, lv in self.samples])
        print(
            f"[{channel}] {split} 분포 / distribution: "
            + " | ".join(f"L{lv}:{split_counts.get(lv,0)}" for lv in range(NUM_LEVELS))
        )

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int):
        img_path, level = self.samples[idx]

        # 이미지 로드 / Load image
        image = cv2.imread(str(img_path))
        if image is None:
            raise ValueError(f"Image not found / 이미지 없음: {img_path}")

        #  기준 전처리 /  standard preprocessing
        image = cv2.resize(image, (IMAGE_SIZE, IMAGE_SIZE))
        image = image / 255.0

        # 학습 시 증강 적용 / Apply augmentation during training
        if self.augment:
            if random.random() > 0.5:
                image = cv2.flip(image, 1)  # 수평 뒤집기 / Horizontal flip
            if random.random() > 0.5:
                value = random.randint(-30, 30) / 255.0
                image = np.clip(image + value, 0, 1)  # 밝기 변동 / Brightness jitter
            if random.random() > 0.5:
                noise = random.randint(0, 10) / 255.0
                image = np.clip(image + noise, 0, 1)  # 노이즈 추가 / Add noise

        # (H, W, 3) → (3, H, W) 텐서 변환 / Convert to tensor
        tensor = torch.tensor(image).permute(2, 0, 1).float()
        return tensor, level


# 데이터셋 구성 / Build datasets
train_ds = CMYKDataset(TARGET_CHANNEL, split="train", augment=True, oversample=True)
val_ds = CMYKDataset(TARGET_CHANNEL, split="val", augment=False, oversample=False)
test_ds = CMYKDataset(TARGET_CHANNEL, split="test", augment=False, oversample=False)

print(
    f"\n[{TARGET_CHANNEL}] Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}"
)

if len(train_ds) == 0:
    print(f"[WARN] No training data / 학습 데이터 없음")
    print(f"       Check / 확인: {LABELED_DIR / TARGET_CHANNEL}")
else:
    train_loader = DataLoader(
        train_ds,
        batch_size=min(BATCH_SIZE, len(train_ds)),
        shuffle=True,
        drop_last=True,  # 마지막 불완전 배치 제거 / Drop last incomplete batch
        num_workers=0,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=min(BATCH_SIZE, max(len(val_ds), 1)),
        shuffle=False,
        num_workers=0,
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=min(BATCH_SIZE, max(len(test_ds), 1)),
        shuffle=False,
        num_workers=0,
    )
    print(
        f"Train batches / 배치 수: {len(train_loader)} | Val batches: {len(val_loader)}"
    )

[C] Oversampling 적용 / Applied: 119개 → 310개
[C] train 분포 / distribution: L0:0 | L1:62 | L2:62 | L3:62 | L4:62 | L5:62
[C] val 분포 / distribution: L0:0 | L1:2 | L2:5 | L3:13 | L4:4 | L5:1
[C] test 분포 / distribution: L0:0 | L1:3 | L2:6 | L3:14 | L4:6 | L5:0

[C] Train: 310 | Val: 25 | Test: 29
Train batches / 배치 수: 19 | Val batches: 2


---
## 4. Build Model / 모델 구성

- Backbone + Classifier MLP Head
- Backbone → Linear(256) → BN → ReLU → Dropout → Linear(6)
- CrossEntropyLoss includes Softmax internally — do NOT add Softmax to Head
- CrossEntropyLoss 는 내부적으로 Softmax를 포함하므로 Head에 Softmax를 추가하지 않는다

In [4]:
class ClassifierHead(nn.Module):
    """
    Supervised Classification 전용 Classifier Head.
    Backbone output → Linear(256) → BN → ReLU → Dropout → Linear(6)
    """

    def __init__(
        self,
        in_dim: int,
        hidden_dim: int = 256,
        num_classes: int = 6,
        dropout: float = 0.3,
    ):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class GrayspotModel(nn.Module):
    """Backbone + ClassifierHead 전체 모델 / Full model."""

    def __init__(self, backbone, feature_dim: int, num_classes: int = 6):
        super().__init__()
        self.backbone = backbone
        self.head = ClassifierHead(feature_dim, num_classes=num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        features = self.backbone(x)
        return self.head(features)


# 1. Backbone 로드 / Load backbone
if BACKBONE_NAME == "efficientnet_b0":
    from torchvision.models import EfficientNet_B0_Weights

    backbone = models.efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
    feature_dim = backbone.classifier[1].in_features  # 1280
    backbone.classifier = nn.Identity()

elif BACKBONE_NAME == "resnet50":
    from torchvision.models import ResNet50_Weights

    backbone = models.resnet50(weights=ResNet50_Weights.DEFAULT)
    feature_dim = backbone.fc.in_features  # 2048
    backbone.fc = nn.Identity()

# 2. 모델 구성 / Build model
model = GrayspotModel(backbone, feature_dim, num_classes=NUM_LEVELS).to(device)

# 3. Phase 0 backbone 로드 (모델 구성 후에 로드) / Load Phase 0 backbone after model is built
if PHASE0_BACKBONE.exists():
    state = torch.load(PHASE0_BACKBONE, map_location=device)
    backbone_state = {
        k.replace("backbone.", ""): v
        for k, v in state.items()
        if k.startswith("backbone.")
    }
    if backbone_state:
        model.backbone.load_state_dict(backbone_state, strict=False)
        print(f"[PASS] Phase 0 backbone 로드 / Loaded: {PHASE0_BACKBONE.name}")
    else:
        print(
            "[WARN] backbone 키 없음 — pretrained weights 유지 / No backbone keys — keeping pretrained weights"
        )
else:
    print(
        "[INFO] Phase 0 backbone 없음 — pretrained weights로 시작 / Starting from pretrained weights"
    )

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model built / 모델 구성 완료: {BACKBONE_NAME}")
print(f"Feature dim / 특징 차원: {feature_dim}")
print(f"Total params / 전체 파라미터: {total_params:,}")
print(f"Trainable params / 학습 파라미터: {trainable_params:,}")

[PASS] Phase 0 backbone 로드 / Loaded: phase0_backbone_C.pt
Model built / 모델 구성 완료: efficientnet_b0
Feature dim / 특징 차원: 1280
Total params / 전체 파라미터: 4,337,538
Trainable params / 학습 파라미터: 4,337,538


---
## 5. Training Loop (10 Epochs) / 학습 루프 (10 에폭)

- Loss: CrossEntropyLoss
- Optimizer: AdamW
- Scheduler: CosineAnnealingLR
- Best model saved based on validation accuracy
- 검증 정확도 기준으로 최적 모델을 저장한다

In [5]:
if len(train_ds) == 0:
    print("[WARN] Skipping training / 학습 건너뜀 — 데이터 없음 / No data")
else:
    criterion = nn.CrossEntropyLoss()  # CE Loss — Softmax 포함 / includes Softmax
    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

    history = []
    best_val_acc = 0.0
    best_epoch = 0
    best_path = MODEL_DIR / f"baseline_{TARGET_CHANNEL}.pt"

    print("=" * 65)
    print(f"  Training Start / 학습 시작 — Channel: [{TARGET_CHANNEL}]")
    print("=" * 65)
    print(
        f'  {"Epoch":<8} {"Train Loss":<14} {"Train Acc":<12} {"Val Loss":<12} {"Val Acc":<10} LR'
    )
    print("-" * 65)

    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()

        # ── Train / 학습 ──
        model.train()  # train 모드 — BN, Dropout 활성화 / Activate BN, Dropout
        train_loss, train_correct, train_total = 0.0, 0, 0

        for x, labels in train_loader:
            x, labels = x.to(device), labels.to(device)
            logits = model(x)
            loss = criterion(logits, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            train_correct += (logits.argmax(1) == labels).sum().item()
            train_total += len(labels)

        train_loss_avg = train_loss / max(len(train_loader), 1)
        train_acc = train_correct / max(train_total, 1)

        # ── Validation / 검증 ──
        model.eval()  # eval 모드 — BN, Dropout 비활성화 / Deactivate BN, Dropout
        val_loss, val_correct, val_total = 0.0, 0, 0

        with torch.no_grad():
            for x, labels in val_loader:
                x, labels = x.to(device), labels.to(device)
                logits = model(x)
                loss = criterion(logits, labels)

                val_loss += loss.item()
                val_correct += (logits.argmax(1) == labels).sum().item()
                val_total += len(labels)

        val_loss_avg = val_loss / max(len(val_loader), 1)
        val_acc = val_correct / max(val_total, 1)

        scheduler.step()
        elapsed = time.time() - t0
        lr = optimizer.param_groups[0]["lr"]

        record = {
            "epoch": epoch,
            "train_loss": round(train_loss_avg, 6),
            "train_acc": round(train_acc, 4),
            "val_loss": round(val_loss_avg, 6),
            "val_acc": round(val_acc, 4),
            "lr": round(lr, 8),
            "elapsed": round(elapsed, 2),
        }
        history.append(record)

        # Best 모델 저장 / Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch
            torch.save(model.state_dict(), best_path)

        print(
            f"  {epoch:<8} {train_loss_avg:<14.4f} {train_acc:<12.4f} "
            f"{val_loss_avg:<12.4f} {val_acc:<10.4f} {lr:.2e}"
        )

    print("-" * 65)
    print(f"  Best Val Acc: {best_val_acc:.4f} (Epoch {best_epoch})")
    print(f"  Model saved / 저장: {best_path}")
    print("=" * 65)

  Training Start / 학습 시작 — Channel: [C]
  Epoch    Train Loss     Train Acc    Val Loss     Val Acc    LR
-----------------------------------------------------------------
  1        1.5993         0.3980       1.5438       0.4400     9.97e-05
  2        1.2006         0.6151       1.2786       0.4400     9.89e-05
  3        1.0463         0.6316       1.2214       0.4800     9.76e-05
  4        0.9247         0.6842       1.1180       0.5200     9.57e-05
  5        0.7573         0.7368       0.9420       0.6000     9.34e-05
  6        0.7656         0.7368       0.8207       0.6400     9.05e-05
  7        0.7155         0.7566       0.7969       0.6400     8.73e-05
  8        0.6583         0.7796       0.7302       0.6800     8.36e-05
  9        0.6176         0.7829       0.7054       0.6400     7.96e-05
  10       0.5799         0.8026       0.6389       0.6800     7.52e-05
  11       0.5506         0.8191       0.6101       0.7200     7.06e-05
  12       0.5251         0.8191    

---
## 6. Verify Results / 결과 확인

- Evaluate on the test set using the best saved model
- 저장된 최적 모델로 테스트셋을 평가한다

In [6]:
if len(train_ds) == 0:
    print("[WARN] No data — skipping / 데이터 없음 — 건너뜀")
else:
    # 학습 이력 요약 / Training history summary
    print("학습 이력 요약 / Training History Summary")
    print(f'  {"Epoch":<8} {"Train Acc":<12} {"Val Acc":<10}')
    print("-" * 32)
    for r in history:
        marker = " <-- best" if r["epoch"] == best_epoch else ""
        print(
            f'  {r["epoch"]:<8} {r["train_acc"]:<12.4f} {r["val_acc"]:<10.4f}{marker}'
        )

    # Best 모델 로드 후 테스트셋 평가 / Load best model and evaluate on test set
    print("\n테스트셋 평가 / Test Set Evaluation")
    print("-" * 40)

    # 디버그: test_ds 샘플 확인 / Debug: check test_ds samples
    print(f"[DEBUG] test_ds 샘플 수 / samples: {len(test_ds)}")
    if len(test_ds) > 0:
        sample_x, sample_y = test_ds[0]
        print(f"[DEBUG] test_ds[0] shape: {sample_x.shape} | level: {sample_y}")
    print(f"[DEBUG] test_loader 배치 수 / batches: {len(test_loader)}")
    print(f"[DEBUG] best_path: {best_path}")
    print(f"[DEBUG] best_path exists: {best_path.exists()}")

    # Best 모델 로드 / Load best model
    model.load_state_dict(torch.load(best_path, map_location=device))
    model.eval()  # eval 모드 필수 / eval mode required
    print(f"[DEBUG] model.training: {model.training}")  # False 여야 정상

    test_correct, test_total = 0, 0
    y_true, y_pred = [], []

    with torch.no_grad():
        for batch_idx, (x, labels) in enumerate(test_loader):
            x, labels = x.to(device), labels.to(device)
            logits = model(x)
            preds = logits.argmax(1)

            if batch_idx == 0:
                print(f"[DEBUG] 첫 배치 labels: {labels.tolist()}")
                print(f"[DEBUG] 첫 배치 preds:  {preds.tolist()}")

            test_correct += (preds == labels).sum().item()
            test_total += len(labels)
            y_true.extend(labels.cpu().tolist())
            y_pred.extend(preds.cpu().tolist())

    test_acc = test_correct / max(test_total, 1)
    mae = sum(abs(t - p) for t, p in zip(y_true, y_pred)) / max(len(y_true), 1)

    print(f"\n  Test Accuracy / 테스트 정확도: {test_acc:.4f}")
    print(f"  MAE (Level Error) / 레벨 오차: {mae:.4f}")
    print(f"  Test samples / 테스트 샘플: {test_total}개")
    print(f"  y_true: {y_true}")
    print(f"  y_pred: {y_pred}")

    # Confusion Matrix (텍스트) / Confusion Matrix (text)
    print("\nConfusion Matrix (Pred→ / True↓):")
    print(f'  {"":>4}' + "".join(f'{"P"+str(i):>6}' for i in range(NUM_LEVELS)))
    cm = [[0] * NUM_LEVELS for _ in range(NUM_LEVELS)]
    for t, p in zip(y_true, y_pred):
        cm[t][p] += 1
    for i, row in enumerate(cm):
        print(f"  T{i}  " + "".join(f"{v:>6}" for v in row))

    # 학습 이력 CSV 저장 / Save training history to CSV
    csv_path = REPORT_DIR / f"baseline_history_{TARGET_CHANNEL}.csv"
    with open(csv_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=history[0].keys())
        writer.writeheader()
        writer.writerows(history)

    print(f"\n학습 이력 저장 / History saved: {csv_path}")
    print(f"모델 저장 / Model saved: {best_path}")

    # 성능 목표 대비 확인 / Check against performance targets
    TARGET_ACC = 0.85
    TARGET_MAE = 0.50
    print("\n성능 목표 대비 / vs. Performance Targets:")
    print(
        f"  Accuracy: {test_acc:.4f} (target >= {TARGET_ACC}) "
        f'{"[PASS]" if test_acc >= TARGET_ACC else "[FAIL]"}'
    )
    print(
        f"  MAE:      {mae:.4f} (target <= {TARGET_MAE}) "
        f'{"[PASS]" if mae <= TARGET_MAE else "[FAIL]"}'
    )

학습 이력 요약 / Training History Summary
  Epoch    Train Acc    Val Acc   
--------------------------------
  1        0.4507       0.4400    
  2        0.6447       0.4800    
  3        0.6678       0.4800    
  4        0.7138       0.4800    
  5        0.7730       0.5600    
  6        0.7533       0.6000    
  7        0.7862       0.6000    
  8        0.7993       0.6000    
  9        0.8059       0.5600    
  10       0.8092       0.7200    
  11       0.8586       0.6400    
  12       0.8553       0.6800    
  13       0.8783       0.8000     <-- best
  14       0.9046       0.7600    
  15       0.8882       0.7200    
  16       0.9342       0.6800    
  17       0.9276       0.7200    
  18       0.9342       0.7200    
  19       0.9770       0.6800    
  20       0.9572       0.7200    
  21       0.9572       0.7600    
  22       0.9178       0.7200    
  23       0.9507       0.7200    
  24       0.9704       0.7600    
  25       0.9572       0.6800    
  26       0